In [1]:
import torch
from ultralytics import YOLO

# ── 0. GPU 확인 ──────────────────────────────────────────
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = 0 if torch.cuda.is_available() else "cpu"   # 0 = 첫 번째 GPU

# ── 1. 모델 로드 ─────────────────────────────────────────
# n/s/m/l/x 중 선택. GPU 메모리 여유 있으면 yolov8s.pt 권장
model = YOLO("yolov8s.pt")          # 사전학습 가중치에서 시작

# ── 2. 학습 ─────────────────────────────────────────────
results = model.train(
    data="wheelchair_datasets/merged/data.yaml",  # ← 통합 data.yaml 경로
    epochs=100,
    imgsz=640,
    batch=16,            # GPU OOM이면 8 또는 4로 낮추기. -1 이면 자동
    device=device,       # GPU 사용
    workers=8,
    patience=20,         # 20 epoch 동안 개선 없으면 조기 종료
    optimizer="auto",
    lr0=0.01,
    cos_lr=True,
    amp=True,            # 혼합정밀 → GPU 속도/메모리 이득
    cache=False,         # RAM 충분하면 True 로 I/O 가속

    # ── 데이터 증강 (전동휠체어 vs 유모차 오분류 완화에 도움) ──
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

    project="runs/wheelchair",
    name="yolov8s_wabs",
    exist_ok=True,
)

# ── 3. 검증 ─────────────────────────────────────────────
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50   :", metrics.box.map50)

# ── 4. 최적 가중치로 추론 테스트 ─────────────────────────
best = YOLO("runs/wheelchair/yolov8s_wabs/weights/best.pt")
# best.predict(source="테스트이미지경로.jpg", save=True, conf=0.25)

/home/jetson/PycharmProjects/wheeling/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


CUDA available: False
Ultralytics 8.4.62 🚀 Python-3.10.12 torch-2.12.0+cu130 CPU (ARMv8 Processor rev 1 (v8l))
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=wheelchair_datasets/merged/data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_wabs, nbs=64, nms=False, opset=None, optimize=False, optimiz

RuntimeError: Dataset 'wheelchair_datasets/merged/data.yaml' error ❌ 'wheelchair_datasets/merged/data.yaml' does not exist